# 🔍 RAG Challenge: Threat Intelligence Retrieval

Build a **Retrieval-Augmented Generation (RAG)** pipeline over the MITRE ATT&CK knowledge base.  
You will chunk real threat data, implement two retrieval strategies (keyword + semantic), compare them, and finally build a **hybrid search** that combines the best of both.

**Dataset:** `enterprise-attack.json` — the full MITRE ATT&CK Enterprise matrix (STIX 2.1 format).

---
## Part 0 — Setup & Dependencies

We need only lightweight, pip-installable libraries — no external APIs required.

In [1]:
import json, re, textwrap
import numpy as np
from collections import Counter

---
## Part 1 — Loading & Understanding the Data

The MITRE ATT&CK `enterprise-attack.json` is a **STIX 2.1 bundle**. It contains many object types (identities, relationships, malware, …), but we care about **`attack-pattern`** objects — these describe adversary techniques.

Each `attack-pattern` has:
| Field | What it contains |
|---|---|
| `name` | Technique name (e.g. *Credential Stuffing*) |
| `description` | Detailed prose describing the behaviour |
| `kill_chain_phases` | Which ATT&CK tactic(s) it maps to |
| `external_references` | Links, MITRE IDs (e.g. T1110.004) |
| `x_mitre_platforms` | Affected platforms |

In [2]:
with open("enterprise-attack.json") as f:
    stix_bundle = json.load(f)

# Filter to attack-pattern objects that are NOT revoked/deprecated
techniques = [
    obj for obj in stix_bundle["objects"]
    if obj.get("type") == "attack-pattern"
    and not obj.get("revoked", False)
    and not obj.get("x_mitre_deprecated", False)
]

print(f"Loaded {len(techniques)} active ATT&CK techniques")
print(f"Example: {techniques[0]['name']}")

Loaded 691 active ATT&CK techniques
Example: Extra Window Memory Injection


---
## Part 2 — Chunking

### Why do we chunk?

RAG retrieval works best when each "document" is a **focused, self-contained piece of text**.  
If documents are too long, the retriever might match on irrelevant sections.  
If they're too short, we lose context.

### Our chunking strategy

Each ATT&CK technique already has a natural boundary — its `name` + `description`. These descriptions range from a single paragraph to several paragraphs. We'll create **one chunk per technique** that combines:

1. **Metadata header** — technique ID, name, tactic, platforms (gives the retriever keyword anchors)
2. **Description body** — the full prose description

This is a **document-level chunking** approach — simple but effective because ATT&CK descriptions are already well-scoped (typically 100–400 words each).

> **When would you split further?** If descriptions were multi-page documents, you'd split into overlapping windows (e.g. 256 tokens with 64-token overlap). For ATT&CK, one-chunk-per-technique is the right granularity.

In [3]:
def get_technique_id(technique):
    """Extract the MITRE technique ID (e.g. T1110.004) from external references."""
    for ref in technique.get("external_references", []):
        if ref.get("source_name") == "mitre-attack":
            return ref.get("external_id", "unknown")
    return "unknown"

def get_tactics(technique):
    """Extract tactic names from kill_chain_phases."""
    return [p["phase_name"] for p in technique.get("kill_chain_phases", [])]

def chunk_technique(technique):
    """Build a single text chunk from an ATT&CK technique object."""
    tid = get_technique_id(technique)
    name = technique.get("name", "")
    desc = technique.get("description", "")
    tactics = ", ".join(get_tactics(technique))
    platforms = ", ".join(technique.get("x_mitre_platforms", []))

    # Remove markdown citation links like (Citation: ...) for cleaner text
    desc_clean = re.sub(r"\(Citation:[^)]*\)", "", desc).strip()

    chunk = (
        f"Technique: {tid} — {name}\n"
        f"Tactics: {tactics}\n"
        f"Platforms: {platforms}\n\n"
        f"{desc_clean}"
    )
    return {"id": tid, "name": name, "tactics": tactics, "text": chunk}

# Build the chunk corpus
chunks = [chunk_technique(t) for t in techniques]

print(f"Created {len(chunks)} chunks\n")
print("=== Example chunk ===")
print(chunks[0]["text"][:600])

Created 691 chunks

=== Example chunk ===
Technique: T1055.011 — Extra Window Memory Injection
Tactics: defense-evasion, privilege-escalation
Platforms: Windows

Adversaries may inject malicious code into process via Extra Window Memory (EWM) in order to evade process-based defenses as well as possibly elevate privileges. EWM injection is a method of executing arbitrary code in the address space of a separate live process. 

Before creating a window, graphical Windows-based processes must prescribe to or register a windows class, which stipulate appearance and behavior (via windows procedures, which are functions that handle input/out


### Quick stats on chunk sizes

In [4]:
word_counts = [len(c["text"].split()) for c in chunks]
print(f"Chunks: {len(word_counts)}")
print(f"Min words: {min(word_counts)}")
print(f"Max words: {max(word_counts)}")
print(f"Median:    {sorted(word_counts)[len(word_counts)//2]}")
print(f"Mean:      {sum(word_counts)/len(word_counts):.0f}")

Chunks: 691
Min words: 42
Max words: 565
Median:    163
Mean:      181


---
## Part 3 — BM25 Keyword Search

### What is BM25?

**BM25** (Best Matching 25) is the gold-standard algorithm for *keyword-based* retrieval. It's what search engines used before neural models, and it's still a strong baseline.

BM25 scores each document against a query using **term frequency (TF)** and **inverse document frequency (IDF)**:

$$\text{score}(D, Q) = \sum_{i=1}^{n} \text{IDF}(q_i) \cdot \frac{f(q_i, D) \cdot (k_1 + 1)}{f(q_i, D) + k_1 \cdot \left(1 - b + b \cdot \frac{|D|}{\text{avgdl}}\right)}$$

In plain English:
- **TF part:** How often does the query term appear in this document? More = better, but with diminishing returns.
- **IDF part:** Is this term rare across all documents? Rare terms are weighted higher ("credential" is more informative than "the").
- **Length normalization:** Longer documents don't get an unfair advantage.

### Strengths & Weaknesses
| ✅ Strengths | ❌ Weaknesses |
|---|---|
| Fast, no GPU needed | Misses synonyms ("creds" ≠ "credentials") |
| Great when the user knows exact terms | No understanding of meaning |
| Interpretable scoring | Can't handle paraphrased queries |

In [5]:
import math

# Tokenize: lowercase and split on non-alphanumeric characters
def tokenize(text):
    return re.findall(r"[a-z0-9]+", text.lower())

class BM25:
    def __init__(self, corpus_tokens, k1=1.5, b=0.75):
        self.k1 = k1
        self.b = b
        self.corpus_size = len(corpus_tokens)
        self.doc_lens = [len(doc) for doc in corpus_tokens]
        self.avgdl = sum(self.doc_lens) / self.corpus_size
        self.doc_freqs = []
        self.idf = {}
        self._build_index(corpus_tokens)

    def _build_index(self, corpus_tokens):
        df = Counter()
        for doc in corpus_tokens:
            tf = Counter(doc)
            self.doc_freqs.append(tf)
            for term in tf:
                df[term] += 1
        for term, freq in df.items():
            self.idf[term] = math.log((self.corpus_size - freq + 0.5) / (freq + 0.5) + 1.0)

    def get_scores(self, query_tokens):
        scores = np.zeros(self.corpus_size)
        for q in query_tokens:
            if q not in self.idf:
                continue
            idf = self.idf[q]
            for i, tf in enumerate(self.doc_freqs):
                f = tf.get(q, 0)
                dl = self.doc_lens[i]
                num = f * (self.k1 + 1)
                denom = f + self.k1 * (1 - self.b + self.b * dl / self.avgdl)
                scores[i] += idf * num / denom
        return scores

# Build the BM25 index over all chunk texts
corpus_tokens = [tokenize(c["text"]) for c in chunks]
bm25 = BM25(corpus_tokens)

print(f"BM25 index built over {len(corpus_tokens)} documents")

BM25 index built over 691 documents


In [6]:
def bm25_search(query, top_k=5):
    """Return the top-k chunks ranked by BM25 score."""
    query_tokens = tokenize(query)
    scores = bm25.get_scores(query_tokens)
    top_indices = np.argsort(scores)[::-1][:top_k]
    results = []
    for idx in top_indices:
        results.append({
            "rank": len(results) + 1,
            "id": chunks[idx]["id"],
            "name": chunks[idx]["name"],
            "score": float(scores[idx]),
            "snippet": chunks[idx]["text"][:200]
        })
    return results

In [8]:
# Test: search for a specific term
query = "credential stuffing"

print(f"BM25 results for: '{query}'\n")
for r in bm25_search(query):
    print(f"  #{r['rank']}  {r['id']:12s}  {r['name']:40s}  score={r['score']:.2f}")

BM25 results for: 'credential stuffing'

  #1  T1110.004     Credential Stuffing                       score=12.75
  #2  T1608.006     SEO Poisoning                             score=4.81
  #3  T1555.004     Windows Credential Manager                score=4.35
  #4  T1003         OS Credential Dumping                     score=3.78
  #5  T1556.008     Network Provider DLL                      score=3.70


### Try more queries

Experiment with different queries to see how BM25 behaves:

In [9]:
test_queries = [
    "brute force login",              # exact terms
    "attacker reuses stolen passwords across services",   # paraphrased — same concept!
    "lateral movement using remote desktop",              # different topic
    "exfiltration of data over DNS tunnel",               # another topic
]

for q in test_queries:
    results = bm25_search(q, top_k=3)
    print(f"Query: '{q}'")
    for r in results:
        print(f"    {r['id']:12s}  {r['name']:40s}  score={r['score']:.2f}")
    print()

Query: 'brute force login'
    T1110         Brute Force                               score=15.55
    T1595.003     Wordlist Scanning                         score=10.61
    T1558.003     Kerberoasting                             score=10.11

Query: 'attacker reuses stolen passwords across services'
    T1606.001     Web Cookies                               score=8.75
    T1110.004     Credential Stuffing                       score=8.03
    T1589.001     Credentials                               score=7.66

Query: 'lateral movement using remote desktop'
    T1021.001     Remote Desktop Protocol                   score=17.01
    T1563.002     RDP Hijacking                             score=16.41
    T1021         Remote Services                           score=15.36

Query: 'exfiltration of data over DNS tunnel'
    T1048.003     Exfiltration Over Unencrypted Non-C2 Protocol  score=15.23
    T1572         Protocol Tunneling                        score=14.76
    T1048         Exfiltr

### Reflection — BM25

1. Notice how the **exact-term** query ("credential stuffing brute force login") works well — BM25 matches the words directly.
2. Now look at the **paraphrased** query ("attacker reuses stolen passwords across services"). Does it still find *Credential Stuffing*? Probably not as the top result, because BM25 doesn't understand that "reuses stolen passwords" **means the same thing** as "credential stuffing".

This is the core limitation of keyword search — it has **no semantic understanding**.

---
## Part 4 — Semantic Search (TF-IDF Vectors as a Lightweight Proxy)

### What is Semantic Search?

True semantic search encodes both queries and documents into **dense vector embeddings** (using models like Sentence-BERT) and retrieves by **cosine similarity** in the embedding space.

To keep this workshop **dependency-light** (no GPU, no large downloads), we'll use **TF-IDF vectors** with cosine similarity, which captures some semantic signal through n-grams and term weighting. This demonstrates the same retrieval pattern — encode → compare vectors → rank by similarity.

> **In production**, you'd replace TF-IDF with a proper embedding model (e.g. `all-MiniLM-L6-v2` from `sentence-transformers`). The retrieval logic stays the same — only the vectoriser changes.

### How TF-IDF Vector Search Works

1. **Fit a TF-IDF vectoriser** on the corpus — each document becomes a sparse high-dimensional vector where each dimension is a weighted term.
2. **Transform the query** into the same vector space.
3. **Cosine similarity** measures the angle between query and document vectors — smaller angle = more similar.

$$\text{cosine\_sim}(\vec{q}, \vec{d}) = \frac{\vec{q} \cdot \vec{d}}{\|\vec{q}\| \cdot \|\vec{d}\|}$$

### Why is this better than raw BM25 for meaning?

Using **ngram ranges** (e.g. bigrams like "brute force", "remote desktop") captures some phrase-level patterns. Words that appear in similar contexts end up with similar TF-IDF profiles, giving a weak form of semantic similarity.

| ✅ Strengths | ❌ Weaknesses |
|---|---|
| Captures some phrase-level similarity | Still mostly lexical |
| Fast, no model download | Not true deep semantic understanding |
| Same API pattern as dense embedding search | Lower quality than transformer embeddings |

In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Build TF-IDF matrix with unigrams + bigrams for richer representation
corpus_texts = [c["text"] for c in chunks]

tfidf = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),    # unigrams + bigrams
    max_df=0.9,            # ignore terms in >90% of docs
    min_df=2,              # ignore terms in <2 docs
    sublinear_tf=True      # apply log normalization to TF
)
tfidf_matrix = tfidf.fit_transform(corpus_texts)

print(f"TF-IDF matrix shape: {tfidf_matrix.shape}  (documents × features)")

TF-IDF matrix shape: (691, 18417)  (documents × features)


In [11]:
def semantic_search(query, top_k=5):
    """Return top-k chunks by cosine similarity in TF-IDF space."""
    query_vec = tfidf.transform([query])
    similarities = cosine_similarity(query_vec, tfidf_matrix).flatten()
    top_indices = np.argsort(similarities)[::-1][:top_k]
    results = []
    for idx in top_indices:
        results.append({
            "rank": len(results) + 1,
            "id": chunks[idx]["id"],
            "name": chunks[idx]["name"],
            "score": float(similarities[idx]),
            "snippet": chunks[idx]["text"][:200]
        })
    return results

In [12]:
# Same queries — compare results to BM25
for q in test_queries:
    results = semantic_search(q, top_k=3)
    print(f"Query: '{q}'")
    for r in results:
        print(f"    {r['id']:12s}  {r['name']:40s}  score={r['score']:.3f}")
    print()

Query: 'brute force login'
    T1110         Brute Force                               score=0.232
    T1595.003     Wordlist Scanning                         score=0.174
    T1201         Password Policy Discovery                 score=0.162

Query: 'attacker reuses stolen passwords across services'
    T1589.001     Credentials                               score=0.113
    T1110.004     Credential Stuffing                       score=0.091
    T1555         Credentials from Password Stores          score=0.064

Query: 'lateral movement using remote desktop'
    T1021.001     Remote Desktop Protocol                   score=0.217
    T1021         Remote Services                           score=0.206
    T1563.002     RDP Hijacking                             score=0.186

Query: 'exfiltration of data over DNS tunnel'
    T1052.001     Exfiltration over USB                     score=0.158
    T1572         Protocol Tunneling                        score=0.146
    T1011         Exfiltrat

### Reflection — Semantic Search

Compare the semantic search results to BM25:
1. Does the paraphrased query ("attacker reuses stolen passwords across services") rank *Credential Stuffing* any differently?
2. Are there results that appear in one method but not the other?

---
## Part 5 — Side-by-Side Comparison

Let's make the differences visible.

In [13]:
def compare_search(query, top_k=5):
    """Run both search methods and display results side by side."""
    bm25_results = bm25_search(query, top_k)
    sem_results = semantic_search(query, top_k)

    print(f"{'='*80}")
    print(f"Query: '{query}'")
    print(f"{'='*80}")
    print(f"\n{'BM25 (Keyword)':^40s} | {'Semantic (TF-IDF)':^40s}")
    print(f"{'-'*40} | {'-'*40}")

    for i in range(top_k):
        bm25_str = f"#{bm25_results[i]['rank']} {bm25_results[i]['id']} {bm25_results[i]['name'][:22]:22s} ({bm25_results[i]['score']:.1f})"
        sem_str  = f"#{sem_results[i]['rank']} {sem_results[i]['id']} {sem_results[i]['name'][:22]:22s} ({sem_results[i]['score']:.3f})"
        print(f"{bm25_str:40s} | {sem_str:40s}")

    # Overlap analysis
    bm25_ids = {r["id"] for r in bm25_results}
    sem_ids  = {r["id"] for r in sem_results}
    overlap  = bm25_ids & sem_ids
    print(f"\nOverlap: {len(overlap)}/{top_k} results appear in both")
    if bm25_ids - sem_ids:
        print(f"  BM25 only:     {bm25_ids - sem_ids}")
    if sem_ids - bm25_ids:
        print(f"  Semantic only: {sem_ids - bm25_ids}")
    print()

In [14]:
compare_search("credential stuffing")
compare_search("attacker reuses stolen passwords across services")
compare_search("lateral movement using remote desktop")
compare_search("data exfiltration over DNS tunnel")

Query: 'credential stuffing'

             BM25 (Keyword)              |            Semantic (TF-IDF)            
---------------------------------------- | ----------------------------------------
#1 T1110.004 Credential Stuffing    (12.7) | #1 T1110.004 Credential Stuffing    (0.161)
#2 T1608.006 SEO Poisoning          (4.8) | #2 T1608.006 SEO Poisoning          (0.071)
#3 T1555.004 Windows Credential Man (4.4) | #3 T1003 OS Credential Dumping  (0.055) 
#4 T1003 OS Credential Dumping  (3.8)    | #4 T1555.004 Windows Credential Man (0.052)
#5 T1556.008 Network Provider DLL   (3.7) | #5 T1556.008 Network Provider DLL   (0.045)

Overlap: 5/5 results appear in both

Query: 'attacker reuses stolen passwords across services'

             BM25 (Keyword)              |            Semantic (TF-IDF)            
---------------------------------------- | ----------------------------------------
#1 T1606.001 Web Cookies            (8.7) | #1 T1589.001 Credentials            (0.113)
#2 T1110.004

---
## Part 6 — 🎯 YOUR CHALLENGE: Hybrid Search

### The Goal

Implement a `hybrid_search(query, top_k, alpha)` function that **combines BM25 and semantic scores** into a single ranking.

### How Hybrid Search Works

The idea is simple: **normalize** both score distributions to `[0, 1]`, then combine them with a tunable weight `alpha`:

$$\text{hybrid\_score}(d) = \alpha \cdot \hat{s}_{\text{BM25}}(d) + (1 - \alpha) \cdot \hat{s}_{\text{semantic}}(d)$$

Where:
- $\hat{s}$ = min-max normalized score (mapped to 0–1)
- $\alpha = 1.0$ → pure BM25 (keyword only)
- $\alpha = 0.0$ → pure semantic search
- $\alpha = 0.5$ → equal weight to both

### Min-Max Normalization

BM25 scores can be 0–30+, while cosine similarities are 0–1. To combine them fairly, normalize each:

$$\hat{s}(d) = \frac{s(d) - s_{\min}}{s_{\max} - s_{\min}}$$

If all scores are identical ($s_{\max} = s_{\min}$), set all normalized scores to 0.

### Your Task

1. Implement `hybrid_search()` in the cell below
2. Test it with the same queries from above
3. Experiment with different `alpha` values and observe how the ranking changes
4. Answer the reflection questions at the end

In [ ]:
def hybrid_search(query, top_k=5, alpha=0.5):
    """
    Combine BM25 and semantic search into a hybrid ranking.

    Parameters
    ----------
    query : str
        The search query.
    top_k : int
        Number of results to return.
    alpha : float
        Weight for BM25 vs semantic. 1.0 = pure BM25, 0.0 = pure semantic.

    Returns
    -------
    list[dict] with keys: rank, id, name, score, bm25_score, semantic_score
    """
    # ─── STEP 1: Get raw scores for ALL documents from both methods ───
    # Hint: for BM25, use bm25.get_scores(tokenize(query))
    # Hint: for semantic, use cosine_similarity(tfidf.transform([query]), tfidf_matrix).flatten()

    # YOUR CODE HERE
    bm25_scores = None      # replace with actual scores (array of length len(chunks))
    sem_scores = None        # replace with actual scores (array of length len(chunks))

    # ─── STEP 2: Normalize both score arrays to [0, 1] using min-max ───
    # Handle the edge case where max == min (set all to 0)

    # YOUR CODE HERE
    bm25_norm = None         # replace with normalized array
    sem_norm = None          # replace with normalized array

    # ─── STEP 3: Compute hybrid score ───
    # hybrid = alpha * bm25_norm + (1 - alpha) * sem_norm

    # YOUR CODE HERE
    hybrid_scores = None     # replace with combined array

    # ─── STEP 4: Rank and return top_k results ───
    # Sort by hybrid_scores descending, take top_k

    # YOUR CODE HERE
    results = []

    return results

### Test your hybrid search

In [ ]:
# Test with alpha=0.5 (equal weight)
query = "credential stuffing brute force login"
print(f"Hybrid results (alpha=0.5) for: '{query}'\n")
for r in hybrid_search(query, top_k=5, alpha=0.5):
    print(f"  #{r['rank']}  {r['id']:12s}  {r['name']:40s}  hybrid={r['score']:.3f}  bm25={r['bm25_score']:.3f}  sem={r['semantic_score']:.3f}")

In [ ]:
# Compare different alpha values on the paraphrased query
query = "attacker reuses stolen passwords across services"

for alpha in [1.0, 0.7, 0.5, 0.3, 0.0]:
    print(f"\nalpha={alpha}  →  {'BM25-heavy' if alpha > 0.5 else 'Semantic-heavy' if alpha < 0.5 else 'Equal'}")
    for r in hybrid_search(query, top_k=3, alpha=alpha):
        print(f"    {r['id']:12s}  {r['name']:40s}  score={r['score']:.3f}")

In [ ]:
# Run all test queries
for q in test_queries:
    print(f"\nQuery: '{q}'")
    for r in hybrid_search(q, top_k=3, alpha=0.5):
        print(f"    {r['id']:12s}  {r['name']:40s}  score={r['score']:.3f}")

---
## Reflection Questions

1. **Is there a single `alpha` that works best for ALL queries?** Or would different query types benefit from different weights?

2. **Beyond the retriever:** In a full RAG system, the retrieved chunks would be passed to an LLM as context. How would retrieval quality affect the LLM's answer about a threat like credential stuffing?

---
## Bonus: Explore the full ATT&CK dataset

Use your hybrid search to explore real threat behaviors. Try queries like:
- `"phishing with malicious attachment"`
- `"adversary uses PowerShell to download malware"`
- `"persistence through scheduled task"`
- `"data encrypted for impact ransomware"`

In [ ]:
# Your exploration here
my_query = ""  # try your own!

if my_query:
    compare_search(my_query)
    print("\nHybrid (alpha=0.5):")
    for r in hybrid_search(my_query, top_k=5, alpha=0.5):
        print(f"  #{r['rank']}  {r['id']:12s}  {r['name']:40s}  score={r['score']:.3f}")